In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Исследование производительности вычисления интеграла\n",
    "**Функция:** $f(x) = \\sin(x)$  \n",
    "**Отрезок:** $[-100, 100]$  \n",
    "**Требуемая точность:** $10^{-4}$"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "dotnet_interactive": {
     "language": "csharp"
    }
   },
   "outputs": [],
   "source": [
    "#r \"nuget: ScottPlot, 5.0.39\"\n",
    "// Если проект называется task13, замени task14 на task13 в пути ниже\n",
    #r "../task15/bin/Debug/net9.0/task15.dll"

    using System.Diagnostics;
    using task15;
    using ScottPlot;
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Определение оптимального шага"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "dotnet_interactive": {
     "language": "csharp"
    }
   },
   "outputs": [],
   "source": [
    "Func<double, double> sin = x => Math.Sin(x);\n",
    "int iterations = 10;\n",
    "double[] steps = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };\n",
    "double requiredAccuracy = 1e-4;\n",
    "\n",
    "double optimalStep = 0;\n",
    "foreach (var step in steps)\n",
    "{\n",
    "    var sw = Stopwatch.StartNew();\n",
    "    double result = 0;\n",
    "    for (int i = 0; i < iterations; i++)\n",
    "        result = SingleThreadIntegral.Solve(-100, 100, sin, step);\n",
    "    sw.Stop();\n",
    "    \n",
    "    double avgTime = sw.ElapsedMilliseconds / (double)iterations;\n",
    "    double error = Math.Abs(result);\n",
    "    \n",
    "    if (error <= requiredAccuracy && optimalStep == 0)\n",
    "        optimalStep = step;\n",
    "        \n",
    "    Console.WriteLine($\"Шаг {step:E2}: время = {avgTime:F2} мс, ошибка = {error:E2}\");\n",
    "}\n",
    "Console.WriteLine($\"\\nОптимальный шаг: {optimalStep:E2}\");"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Зависимость времени от числа потоков"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "dotnet_interactive": {
     "language": "csharp"
    }
   },
   "outputs": [],
   "source": [
    "int[] threadCounts = { 1, 2, 4, 8, 16, 32 };\n",
    "var threadTimes = new List<(int threads, double time)>();\n",
    "\n",
    "foreach (var threads in threadCounts)\n",
    "{\n",
    "    var sw = Stopwatch.StartNew();\n",
    "    for (int i = 0; i < iterations; i++)\n",
    "        DefiniteIntegral.Solve(-100, 100, sin, optimalStep, threads);\n",
    "    sw.Stop();\n",
    "    \n",
    "    double avgTime = sw.ElapsedMilliseconds / (double)iterations;\n",
    "    threadTimes.Add((threads, avgTime));\n",
    "    Console.WriteLine($\"Потоков: {threads,2}, время: {avgTime:F2} мс\");\n",
    "}\n",
    "\n",
    "var plt = new ScottPlot.Plot();\n",
    "var xs = threadTimes.Select(t => (double)t.threads).ToArray();\n",
    "var ys = threadTimes.Select(t => t.time).ToArray();\n",
    "\n",
    "var scatter = plt.Add.Scatter(xs, ys);\n",
    "scatter.MarkerSize = 8;\n",
    "scatter.LineWidth = 2;\n",
    "\n",
    "plt.Title(\"Зависимость времени вычисления от числа потоков\");\n",
    "plt.XLabel(\"Количество потоков\");\n",
    "plt.YLabel(\"Время выполнения, мс\");\n",
    "plt.Grid.Enable();\n",
    "\n",
    "plt.Show(); // Покажет график прямо в ноутбуке"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": ".NET (C#)",
   "language": "C#",
   "name": ".net-csharp"
  },
  "language_info": {
   "name": "C#"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}